# Soil-To-Plant Genus Flow (Rare / Accessory / Core)

Redo of the genus-level Sankey from `rare_species_auroc_timepoint_prediction_full_dataset.ipynb` using the recalled-genera table (`lic_16S_gen_rab_clean.csv`).

Genus classes follow `core_and_rare_species_definitions_and_temporal_patterns.ipynb`. They are computed separately for plant and soil, using the seasonal samples only:

- **Core**: prevalence > 80% of samples
- **Rare**: prevalence < 10% of samples
- **Accessory**: everything in between

Prevalence = # unique samples where the genus is present / total unique samples.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from scipy.stats import fisher_exact

pd.options.display.max_columns = 50

In [2]:
microbiome_abundance = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/Sept_2026_recalled_genera/lic_16S_gen_rab_clean.csv"
)
microbiome_abundance = microbiome_abundance.rename(columns={"Sample": "sampID"})

long_term_microbiome = microbiome_abundance.loc[
    (microbiome_abundance["sample.type"] == "plant")
    & (microbiome_abundance["seascirc"] == "seasonal")
]
long_term_soil_microbiome = microbiome_abundance.loc[
    (microbiome_abundance["sample.type"] == "soil")
    & (microbiome_abundance["seascirc"] == "seasonal")
]

print("plant samples:", long_term_microbiome["sampID"].nunique())
print("soil samples:", long_term_soil_microbiome["sampID"].nunique())

plant samples: 222
soil samples: 28


In [3]:
core_threshold = 0.80
rare_threshold = 0.10
unassigned_labels = {"unassigned", "na", "none", ""}


def classify_genera(df, sample_col="sampID", tax_col="Genus"):
    """Prevalence-based Rare / Accessory / Core classes (same rules as
    core_and_rare_species_definitions_and_temporal_patterns.ipynb)."""
    total_samples = df[sample_col].dropna().nunique()

    working = df[[sample_col, tax_col]].copy()
    working[tax_col] = working[tax_col].astype("string").str.strip()
    working = working[
        working[tax_col].notna()
        & ~working[tax_col].str.casefold().isin(unassigned_labels)
    ]

    out = (
        working.groupby(tax_col)[sample_col]
        .nunique()
        .rename("n_samples_present")
        .to_frame()
    )
    out["total_samples"] = total_samples
    out["prevalence"] = out["n_samples_present"] / total_samples
    out["classification"] = np.select(
        [out["prevalence"] > core_threshold, out["prevalence"] < rare_threshold],
        ["Core", "Rare"],
        default="Accessory",
    )
    return out.sort_values("prevalence", ascending=False)


plant_genus_classification = classify_genera(long_term_microbiome)
soil_genus_classification = classify_genera(long_term_soil_microbiome)

class_counts = pd.DataFrame(
    {
        "plant": plant_genus_classification["classification"].value_counts(),
        "soil": soil_genus_classification["classification"].value_counts(),
    }
).reindex(["Rare", "Accessory", "Core"])
class_counts

,plant,soil
classification,,
Rare,145,77
Accessory,82,119
Core,36,104


## Soil-To-Plant Flow

Tracks each soil genus into one of four plant outcomes: Plant Rare, Plant Accessory, Plant Core, or Not Present in Plants. Genera found in plants but not in soil are excluded.

Fisher's exact test (one-sided, `greater`) checks whether each soil class is enriched in each plant outcome, with all soil genera as the background.

In [4]:
soil_classes = ["Rare", "Accessory", "Core"]
plant_classes = ["Rare", "Accessory", "Core", "Not Present in Plants"]

soil_to_plant_genus_flow = (
    soil_genus_classification[["classification"]]
    .rename(columns={"classification": "soil_classification"})
    .join(
        plant_genus_classification[["classification"]].rename(
            columns={"classification": "plant_classification"}
        ),
        how="left",
    )
    .assign(
        plant_classification=lambda df: df["plant_classification"].fillna(
            "Not Present in Plants"
        )
    )
    .reset_index(names="Genus")
)

genus_flow_counts = soil_to_plant_genus_flow.groupby(
    ["soil_classification", "plant_classification"], as_index=False
).agg(genus_count=("Genus", "nunique"))

genus_flow_counts.pivot(
    index="soil_classification",
    columns="plant_classification",
    values="genus_count",
).reindex(index=soil_classes, columns=plant_classes).fillna(0).astype(int)

plant_classification,Rare,Accessory,Core,Not Present in Plants
soil_classification,,,,
Rare,22,6,0,49
Accessory,50,23,3,43
Core,20,48,33,3


In [5]:
genus_background_total = len(soil_to_plant_genus_flow)
soil_genus_class_totals = soil_to_plant_genus_flow["soil_classification"].value_counts()
plant_genus_class_totals = soil_to_plant_genus_flow[
    "plant_classification"
].value_counts()

genus_enrichment_records = []

for soil_class in soil_classes:
    for plant_class in plant_classes:
        overlap = int(
            genus_flow_counts.loc[
                genus_flow_counts["soil_classification"].eq(soil_class)
                & genus_flow_counts["plant_classification"].eq(plant_class),
                "genus_count",
            ].sum()
        )
        soil_total = int(soil_genus_class_totals.get(soil_class, 0))
        plant_total = int(plant_genus_class_totals.get(plant_class, 0))
        soil_not_plant = soil_total - overlap
        plant_not_soil = plant_total - overlap
        neither = genus_background_total - overlap - soil_not_plant - plant_not_soil

        odds_ratio, p_value = fisher_exact(
            [[overlap, soil_not_plant], [plant_not_soil, neither]],
            alternative="greater",
        )

        genus_enrichment_records.append(
            {
                "soil_classification": soil_class,
                "plant_classification": plant_class,
                "overlap_genera": overlap,
                "soil_class_total": soil_total,
                "plant_class_total": plant_total,
                "background_total_soil_genera": genus_background_total,
                "odds_ratio": round(odds_ratio, 3),
                "p_value_greater": p_value,
            }
        )

genus_enrichment_results = pd.DataFrame(genus_enrichment_records)
genus_enrichment_results

,soil_classification,plant_classification,overlap_genera,soil_class_total,plant_class_total,background_total_soil_genera,odds_ratio,p_value_greater
0,Rare,Rare,22,77,92,300,0.874,7.256408e-01
1,Rare,Accessory,6,77,77,300,0.181,9.999987e-01
2,Rare,Core,0,77,36,300,0.000,1.000000e+00
3,Rare,Not Present in Plants,49,77,95,300,6.734,1.086274e-11
4,Accessory,Rare,50,119,92,300,2.398,4.636987e-04
5,Accessory,Accessory,23,119,77,300,0.563,9.859770e-01
6,Accessory,Core,3,119,36,300,0.116,9.999988e-01
7,Accessory,Not Present in Plants,43,119,95,300,1.404,1.110886e-01
8,Core,Rare,20,104,92,300,0.410,9.995748e-01
9,Core,Accessory,48,104,77,300,4.936,6.555415e-09


In [6]:
class_colors = {
    "Rare": "#d97d54",
    "Accessory": "#7a9e7e",
    "Core": "#30475e",
    "Not Present in Plants": "#b0b7c3",
}


def hex_to_rgba(hex_color, alpha):
    r, g, b = (int(hex_color.lstrip("#")[i : i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r}, {g}, {b}, {alpha})"


def make_sankey(
    flow_df,
    node_labels,
    title_text,
    width=950,
    height=550,
    show_node_counts=False,
):
    """flow_df needs source, target, soil_classification, plant_classification
    and genus_count columns. Nodes are coloured by class; links by soil class."""
    node_index = {label: index for index, label in enumerate(node_labels)}
    display_labels = node_labels

    source_counts = flow_df.groupby("source")["genus_count"].sum()
    target_counts = flow_df.groupby("target")["genus_count"].sum()
    node_totals = pd.concat([source_counts, target_counts], axis=1).max(axis=1)
    node_totals = node_totals.reindex(node_labels, fill_value=0).astype(int)
    if show_node_counts:
        display_labels = [f"{label} (n={node_totals[label]})" for label in node_labels]

    # Pin nodes top-to-bottom in node_labels order (soil left, plant right);
    # plotly's automatic layout otherwise shuffles them
    node_x, node_y = [], []
    for side in ("Soil ", None):
        column = [
            label for label in node_labels if label.startswith("Soil ") == bool(side)
        ]
        heights = node_totals[column] / node_totals[column].sum()
        gap = 0.04
        scale = 1 - gap * (len(column) - 1)
        tops = np.concatenate([[0], np.cumsum(heights * scale + gap)[:-1]])
        for label, top, node_height in zip(column, tops, heights * scale):
            node_x.append(0.001 if side else 0.999)
            node_y.append(top + node_height / 2)
    node_order = [label for label in node_labels if label.startswith("Soil ")] + [
        label for label in node_labels if not label.startswith("Soil ")
    ]
    position = dict(zip(node_order, zip(node_x, node_y)))

    node_colors = [
        class_colors[label.removeprefix("Soil ").removeprefix("Plant ")]
        for label in node_labels
    ]
    link_colors = [
        hex_to_rgba(class_colors[c], 0.35) for c in flow_df["soil_classification"]
    ]

    figure = go.Figure(
        data=[
            go.Sankey(
                arrangement="fixed",
                node=dict(
                    label=display_labels,
                    x=[position[label][0] for label in node_labels],
                    y=[position[label][1] for label in node_labels],
                    pad=18,
                    thickness=22,
                    color=node_colors,
                ),
                link=dict(
                    source=flow_df["source"].map(node_index).tolist(),
                    target=flow_df["target"].map(node_index).tolist(),
                    value=flow_df["genus_count"].tolist(),
                    color=link_colors,
                    label=(
                        flow_df["soil_classification"]
                        + " -> "
                        + flow_df["plant_classification"]
                        + " | genera: "
                        + flow_df["genus_count"].astype(str)
                    ).tolist(),
                ),
            )
        ]
    )
    figure.update_layout(
        title_text=title_text,
        font_size=12,
        width=width,
        height=height,
    )
    return figure


def add_node_names(flow_df):
    flow_df = flow_df.copy()
    flow_df["source"] = "Soil " + flow_df["soil_classification"]
    flow_df["target"] = flow_df["plant_classification"].where(
        flow_df["plant_classification"].eq("Not Present in Plants"),
        "Plant " + flow_df["plant_classification"],
    )
    return flow_df

In [7]:
all_genera_node_labels = (
    [f"Soil {c}" for c in soil_classes]
    + [f"Plant {c}" for c in soil_classes]
    + ["Not Present in Plants"]
)

all_genera_flow = add_node_names(genus_flow_counts)

all_genera_fig = make_sankey(
    all_genera_flow,
    all_genera_node_labels,
    "Flow of Soil Genera Into Plant Rare, Accessory, Core, or Absent Classes",
    width=1100,
    show_node_counts=True,
)
genus_output_path = Path(
    "/Users/michael/Git/Outdoor_microbiome/Recall_genera_update_paper_results/Figures/soil_to_plant_genus_flow_rare_accessory_core.pdf"
)
genus_output_path.parent.mkdir(parents=True, exist_ok=True)
all_genera_fig.write_image(genus_output_path)
print(f"Saved genus-level Sankey figure to {genus_output_path}")
all_genera_fig.show()

Saved genus-level Sankey figure to /Users/michael/Git/Outdoor_microbiome/Recall_genera_update_paper_results/Figures/soil_to_plant_genus_flow_rare_accessory_core.pdf


### Shared genera only

Restricted to genera found in both soil and plants (drops `Not Present in Plants`).

In [8]:
shared_genera_flow = add_node_names(
    genus_flow_counts.loc[
        genus_flow_counts["plant_classification"].ne("Not Present in Plants")
    ]
)

display(
    shared_genera_flow.pivot(
        index="soil_classification",
        columns="plant_classification",
        values="genus_count",
    )
    .reindex(index=soil_classes, columns=soil_classes)
    .fillna(0)
    .astype(int)
)

shared_genera_node_labels = all_genera_node_labels[:-1]

shared_genera_fig = make_sankey(
    shared_genera_flow,
    shared_genera_node_labels,
    "Flow of Shared Soil Genera Into Plant Rare, Accessory, or Core Classes",
    show_node_counts=True,
)
shared_genera_fig.show()

plant_classification,Rare,Accessory,Core
soil_classification,,,
Rare,22,6,0
Accessory,50,23,3
Core,20,48,33


### Rare and core only

Restricted to genera that are rare or core in soil **and** rare or core in plants (drops Accessory on both sides and `Not Present in Plants`).

In [9]:
rare_core_classes = ["Rare", "Core"]

rare_core_genera_flow = add_node_names(
    genus_flow_counts.loc[
        genus_flow_counts["soil_classification"].isin(rare_core_classes)
        & genus_flow_counts["plant_classification"].isin(rare_core_classes)
    ]
)

display(
    rare_core_genera_flow.pivot(
        index="soil_classification",
        columns="plant_classification",
        values="genus_count",
    )
    .reindex(index=rare_core_classes, columns=rare_core_classes)
    .fillna(0)
    .astype(int)
)

rare_core_node_labels = [f"Soil {c}" for c in rare_core_classes] + [
    f"Plant {c}" for c in rare_core_classes
]

rare_core_genera_fig = make_sankey(
    rare_core_genera_flow,
    rare_core_node_labels,
    "Flow of Soil Rare and Core Genera Into Plant Rare or Core Classes",
    show_node_counts=True,
)
rare_core_output_path = genus_output_path.with_name(
    "soil_to_plant_genus_flow_rare_core_only.pdf"
)
rare_core_genera_fig.write_image(rare_core_output_path)
print(f"Saved rare/core-only Sankey figure to {rare_core_output_path}")
rare_core_genera_fig.show()

plant_classification,Rare,Core
soil_classification,,
Rare,22,0
Core,20,33


Saved rare/core-only Sankey figure to /Users/michael/Git/Outdoor_microbiome/Recall_genera_update_paper_results/Figures/soil_to_plant_genus_flow_rare_core_only.pdf
